In [ ]:
# import packages
from tifffile import TiffFile
from tifffile import imread
import matplotlib.pyplot as plt
import numpy as np
import cv2
from skimage.transform import rescale
from tqdm import tqdm  
import pandas as pd
import time
import cupy as cp
import pickle
import types
import os
import scipy
import psutil
import itertools

# from concurrent.futures import ThreadPoolExecutor  
# from multiprocessing import Pool 
import seaborn as sns
import scanpy as sc

import importlib
import my_functions
importlib.reload(my_functions)

sc.set_figure_params(dpi=80)

# %load_ext line_profiler # only needed for profiling functions

plt.rcParams['axes.grid'] = False
fontsize = 16
plt.rcParams['xtick.labelsize'] = fontsize
plt.rcParams['ytick.labelsize'] = fontsize
plt.rcParams['axes.labelsize'] = fontsize
plt.rcParams['axes.labelsize'] = fontsize
plt.rcParams['axes.titlesize'] = fontsize

In [ ]:
def ismember(a, b):
    bind = {}
    for i, elt in enumerate(b):
        if elt not in bind:
            bind[elt] = i
    result = np.array([bind.get(itm, None) for itm in a])  # None can be replaced by any other "not in b" value
    result = [result, np.array([not x is None for x in result])]
    return result

def write_object(myObject, file):
    import dill
    with open(file, 'wb') as f:
        f.write(dill.dumps(myObject))

def read_object(file):
    import dill
    with open(file, 'rb') as f:
        myObject = dill.loads(f.read())
    return myObject

def flatten(my_list):
    return(list(itertools.chain(*my_list)))
    
def grep(pattern, my_list):
    valid_entries = np.array([pattern in x for x in my_list])
    return np.array(my_list)[valid_entries]
    
def grep_exclude(pattern, my_list):
    valid_entries = np.array([pattern in x for x in my_list])
    return np.array(my_list)[np.logical_not(valid_entries)]

In [ ]:
import matplotlib.patches as patches
from shapely import Polygon, Point, affinity, intersection, MultiPolygon
from geopandas.geodataframe import GeoDataFrame
import geopandas as gpd

def mask_to_shapely_polygons(binary_mask):
    """
    Converts a binary mask (NumPy array) into a Shapely Polygon object.

    Args:
        binary_mask (np.ndarray): A 2D NumPy array representing the binary mask,
                                   where non-zero values indicate the object.

    Returns:
        geopandas.geodataframe.GeoDataFrame: A GeoDataFrame with shapely polygons as geometry
    """
    mask_uint8 = binary_mask.astype(np.uint8) * 255
    contours, _ = cv2.findContours(mask_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        return None  # No contours found

    polygons = []
    for contour in contours:
        if contour.shape[0] >= 3:  # A polygon needs at least 3 points
            # Reshape contour to (N, 2) for Shapely Polygon
            coords = contour.squeeze().tolist()
            polygon = Polygon(coords)
            polygons.append(polygon)
    gdf = gpd.GeoDataFrame(geometry=polygons)
    return gdf

### Import dataset

In [ ]:
adata = sc.read_h5ad("../preprocessed/04_adata.h5ad")


In [ ]:
from tifffile import tiffcomment
import xmltodict
filename = '../raw_data/SLIDE-0265_1.0.4_R000_DAPI__FINAL_F.ome.tif'
ome_xml = tiffcomment(filename)
my_dict = xmltodict.parse(ome_xml)
pixel_size = float(my_dict['OME']['Image']['Pixels']['@PhysicalSizeX'])

adata.obs_names = [str(x) + "_" + str(y) for x,y in zip(adata.obs['core_id'], adata.obs['cell_id'])]

adata.obsm["X_spatial_px"] = adata.obs[['centroid_col_global', 'centroid_row_global']].to_numpy() * pixel_size
adata.obsm["X_spatial_px"][:,1] = -adata.obsm["X_spatial_px"][:,1]


Import censor regions

In [ ]:
possible_files = os.listdir("../manual_masks/")
possible_files = grep("censor", possible_files)
possible_files

array(['SLIDE-0265_7.0.4_R000_FITC_pERK-D13-14-4E-488_FINAL_AFR_F__censor_mask.ome.tif',
       'SLIDE-0265_12.0.4_R000_FITC_Ecad-24E10-488_FINAL_AFR_F__censor_mask.ome.tif',
       'SLIDE-0265_13.0.4_R000_Cy3_b-cat-555_FINAL_AFR_F__censor_mask.ome.tif',
       'SLIDE-0265_2.0.4_R000_FITC_KRT17-EP1623-488_FINAL_AFR_F__censor_mask.ome.tif'],
      dtype='<U78')

In [ ]:
censor_file = [
    'SLIDE-0265_12.0.4_R000_FITC_Ecad-24E10-488_FINAL_AFR_F__censor_mask.ome.tif',
    'SLIDE-0265_13.0.4_R000_Cy3_b-cat-555_FINAL_AFR_F__censor_mask.ome.tif',
    'SLIDE-0265_7.0.4_R000_FITC_pERK-D13-14-4E-488_FINAL_AFR_F__censor_mask.ome.tif',
    'SLIDE-0265_2.0.4_R000_FITC_KRT17-EP1623-488_FINAL_AFR_F__censor_mask.ome.tif'
]
censor_nickname = ['Ecad', 'bcat', 'pERK', 'KRT17']

In [ ]:
cell_centroids = gpd.GeoDataFrame(geometry = [Point(x,y) for x,y in zip(adata.obs['centroid_col_global'], adata.obs['centroid_row_global'])])
for my_file, my_nickname in zip(censor_file, censor_nickname):
    IM = imread(os.path.join('../manual_masks/', my_file))
    gdf = mask_to_shapely_polygons((IM > 0).astype(np.uint8))
    joined = cell_centroids.sjoin(gdf)
    censored = np.array([False] * adata.n_obs, dtype = bool)
    censored[joined.index.to_numpy()] = True
    adata.obs['censored_' + my_nickname] = censored.copy()

In [ ]:
adata.obs['censored_0'] = adata.obs['censored_bcat'].copy()
adata.obs['censored_1'] = (adata.obs['censored_bcat']) | (adata.obs['censored_Ecad']) | (adata.obs['censored_pERK'])

In [ ]:
sc.write("../preprocessed/05_adata.h5ad", adata = adata)